# CEO Reputational Risk, Corporate Events & Quantitative Portfolio Risk Analysis

A modular, reproducible quantitative research framework investigating:
1. Corporate event studies and abnormal return modeling.
2. Financial econometrics (GARCH(1,1), downside semivariance, rolling beta).
3. Portfolio risk attribution using Euler's homogeneous risk decomposition.
4. Drawdown episode segmentation and tail-risk metrics (VaR, Expected Shortfall, Cornish-Fisher).
5. Downside stress testing via correlated Monte Carlo simulations.


## 1. Environment & Modular Imports
All core quantitative functions are imported from the production `src/` library.


In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.io as pio

# Set root path to import from src/
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_collection import download_market_data, validate_data_quality
from src.data_processing import (
    compute_returns, compute_indexed_wealth, compute_volume_surprise,
    compute_rolling_beta, generate_cross_asset_summary
)
from src.volatility_models import (
    compute_rolling_volatility, compute_downside_semivolatility, fit_garch_model
)
from src.portfolio_optimization import (
    validate_portfolio_weights, compute_portfolio_returns,
    compute_euler_risk_contribution, calculate_portfolio_performance_metrics,
    compute_monthly_returns, generate_markowitz_efficient_frontier
)
from src.risk_metrics import (
    compute_drawdown_series, identify_drawdown_episodes,
    compute_var_and_es, compute_rolling_var_es
)
from src.stress_testing import (
    run_deterministic_stress, run_idiosyncratic_stress,
    run_correlated_monte_carlo, run_scenario_sensitivity_matrix
)
from src.visualization import (
    plot_historical_prices, plot_normalized_performance,
    plot_returns_distribution, plot_rolling_volatility_beta, plot_garch_volatility,
    plot_correlation_heatmap, plot_risk_reward_bubble, plot_efficient_frontier,
    plot_risk_contribution_comparison, plot_portfolio_equity_curve,
    plot_underwater_drawdowns, plot_rolling_var_es, plot_monthly_heatmap,
    plot_monte_carlo_loss_distribution, plot_stress_test_scenarios,
    plot_event_study_template
)

pio.renderers.default = "notebook"
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
print("Quantitative research environment initialized successfully.")


## 2. Market Data Acquisition & Quality Control
We acquire historical daily OHLCV data for large-cap tech equities and benchmarks from Yahoo Finance with automated local caching.


In [ ]:
STOCKS = ["NVDA", "META", "AMZN", "GOOGL", "TSLA", "AMD", "AVGO", "MSFT"]
BENCHMARKS = ["SPY", "QQQ", "SOXX"]
TICKERS = STOCKS + BENCHMARKS
START_DATE = "2025-11-01"
END_DATE = "2026-11-15"

frames, source_info = download_market_data(
    TICKERS, start=START_DATE, end=END_DATE, cache_dir=PROJECT_ROOT / "data" / "raw"
)
prices, volume = frames["close"], frames["volume"]

quality_report = validate_data_quality(prices)
display(quality_report)
print(f"Data Source: {source_info} | {prices.index.min().date()} to {prices.index.max().date()} | {len(prices)} sessions")


## 3. Financial Returns Transformation
We calculate arithmetic daily returns and evaluate baseline summary statistics across the asset universe.


In [ ]:
returns, log_returns = compute_returns(prices, drop_threshold_tickers=["NVDA", "SPY"])
drawdowns_all = pd.DataFrame({col: compute_drawdown_series(returns[col])[1] for col in returns.columns})
summary_table = generate_cross_asset_summary(returns, drawdowns_all)
display(summary_table.style.format({
    "annualized_return": "{:.2%}",
    "annualized_volatility": "{:.2%}",
    "return_vol_ratio": "{:.2f}",
    "worst_day": "{:.2%}",
    "daily_5pct_quantile": "{:.2%}",
    "max_drawdown": "{:.2%}"
}))


## 4. Asset Performance & Return Distributions
Interactive charts evaluating normalized asset performance ($100 base) and return distributions with 5th-percentile downside tails.


In [ ]:
fig_norm = plot_normalized_performance(prices, focus_tickers=["NVDA", "META", "AMZN", "TSLA", "MSFT", "SPY"])
fig_norm.show()

fig_dist = plot_returns_distribution(returns, focus_tickers=["NVDA", "META", "AMZN", "TSLA", "MSFT"])
fig_dist.show()


## 5. Event Study Verification Gate
The event study framework requires primary source URLs, exact publication timestamps, and market session mapping.
**Methodological constraint:** Because provisional event records currently lack independent source verification, event study outputs remain in standby mode.


In [ ]:
# Provisional event ledger
events = pd.DataFrame([
    dict(event_id="E1", event="Leadership / Governance Inquiry (Provisional)",
         announcement_date="2026-11-08", day0="2026-11-09",
         source_name="", source_url="", verified=False, include=False,
         notes="Pending verification of original publication timestamp."),
    dict(event_id="E2", event="Board Communication / Public Response (Provisional)",
         announcement_date="2026-11-12", day0="2026-11-12",
         source_name="", source_url="", verified=False, include=False,
         notes="Pending confirmation of release relative to market close.")
])

events["source_present"] = events["source_url"].fillna("").str.startswith(("https://", "http://"))
events["analysis_ready"] = events["verified"].astype(bool) & events["include"].astype(bool) & events["source_present"]
display(events)

if not events["analysis_ready"].any():
    print("EVENT STUDY STATUS: No verified event records enabled. Analysis standby preserved.")
    fig_event = plot_event_study_template()
    fig_event.show()


## 6. Volatility Dynamics & GARCH(1,1) Econometrics
We model time-varying conditional volatility for NVIDIA (NVDA) using a GARCH(1,1) specification with Student's t innovations.


In [ ]:
vol20 = compute_rolling_volatility(returns["NVDA"], window=20)
vol60 = compute_rolling_volatility(returns["NVDA"], window=60)
downside20 = compute_downside_semivolatility(returns["NVDA"], window=20)
beta60 = compute_rolling_beta(returns["NVDA"], returns["SPY"], window=60)

fig_vol = plot_rolling_volatility_beta(vol20, vol60, downside20, beta60, target="NVDA")
fig_vol.show()

garch_fit = fit_garch_model(returns["NVDA"])
print(f"GARCH(1,1) Alpha: {garch_fit['alpha']:.4f} | Beta: {garch_fit['beta']:.4f}")
print(f"Volatility Persistence (α + β): {garch_fit['persistence']:.4f}")
print(f"Shock Half-Life: {garch_fit['half_life']:.1f} sessions")

fig_garch = plot_garch_volatility(garch_fit["conditional_volatility"], garch_fit["persistence"], target="NVDA")
fig_garch.show()


## 7. Cross-Asset Correlation & Risk-Reward Positioning


In [ ]:
corr_matrix = returns[STOCKS].corr()
fig_corr = plot_correlation_heatmap(corr_matrix)
fig_corr.show()

fig_bubble = plot_risk_reward_bubble(summary_table, target="NVDA")
fig_bubble.show()


## 8. Portfolio Construction & Euler Risk Attribution
Target Allocation:
- **NVDA**: 30.0%
- **META**: 20.0%
- **AMZN**: 20.0%
- **TSLA**: 15.0%
- **MSFT**: 15.0%

Zero-weight stocks in the expanded universe (GOOGL, AMD, AVGO) are strictly excluded from holdings.
Euler's decomposition partitions portfolio volatility into marginal and percentage component risk contributions:
$$\sigma_p = \sum_{i=1}^N w_i \frac{(\Sigma w)_i}{\sigma_p}, \quad \sum_{i=1}^N PCR_i = 100\%$$


In [ ]:
weights = validate_portfolio_weights({
    "NVDA": 0.30,
    "META": 0.20,
    "AMZN": 0.20,
    "TSLA": 0.15,
    "MSFT": 0.15
})

port_returns = compute_portfolio_returns(returns, weights)
port_metrics = calculate_portfolio_performance_metrics(port_returns)
risk_decomp = compute_euler_risk_contribution(returns, weights)

print(f"Portfolio Total Return: {port_metrics['total_return']:.2%}")
print(f"Portfolio Annualized Volatility: {port_metrics['annualized_volatility']:.2%}")
print(f"Portfolio Sharpe Ratio: {port_metrics['sharpe_ratio']:.2f}")

display(risk_decomp.style.format({
    "weight": "{:.2%}",
    "marginal_risk_contribution": "{:.4f}",
    "absolute_risk_contribution": "{:.4f}",
    "percentage_risk_contribution": "{:.2%}",
    "risk_to_weight_ratio": "{:.2f}"
}))

fig_rc = plot_risk_contribution_comparison(risk_decomp)
fig_rc.show()


## 9. Portfolio Equity Curve & Drawdown Decomposition
We compute cumulative wealth and evaluate underwater drawdown cycles.
**Methodological note:** We isolate distinct economic drawdown episodes from peak to full recovery, separating them from repeated threshold crossings.


In [ ]:
port_wealth, port_drawdowns = compute_drawdown_series(port_returns)
spy_wealth = (1.0 + returns["SPY"]).cumprod()

fig_eq = plot_portfolio_equity_curve(port_wealth, spy_wealth)
fig_eq.show()

top_episodes = identify_drawdown_episodes(port_wealth, top_n=5)
display(top_episodes.style.format({
    "max_drawdown": "{:.2%}",
    "peak_date": lambda d: d.strftime('%Y-%m-%d') if pd.notna(d) else "N/A",
    "trough_date": lambda d: d.strftime('%Y-%m-%d') if pd.notna(d) else "N/A",
    "recovery_date": lambda d: d.strftime('%Y-%m-%d') if pd.notna(d) else "Unrecovered"
}))

fig_dd = plot_underwater_drawdowns(port_drawdowns, top_episodes)
fig_dd.show()


## 10. Downside Tail Risk: VaR & Expected Shortfall
We evaluate 1-day Value-at-Risk (VaR) and Expected Shortfall (ES / CVaR) under:
1. Historical non-parametric empirical quantiles
2. Parametric Gaussian distribution
3. Cornish-Fisher expansion incorporating skewness and excess kurtosis


In [ ]:
tail_risk_95 = compute_var_and_es(port_returns, alpha=0.05)
tail_risk_99 = compute_var_and_es(port_returns, alpha=0.01)

risk_comparison = pd.DataFrame([
    {
        "Confidence": "95%",
        "Historical VaR": tail_risk_95["historical_var"],
        "Parametric VaR": tail_risk_95["parametric_var"],
        "Cornish-Fisher VaR": tail_risk_95["cornish_fisher_var"],
        "Historical ES (CVaR)": tail_risk_95["historical_es"],
        "Parametric ES": tail_risk_95["parametric_es"]
    },
    {
        "Confidence": "99%",
        "Historical VaR": tail_risk_99["historical_var"],
        "Parametric VaR": tail_risk_99["parametric_var"],
        "Cornish-Fisher VaR": tail_risk_99["cornish_fisher_var"],
        "Historical ES (CVaR)": tail_risk_99["historical_es"],
        "Parametric ES": tail_risk_99["parametric_es"]
    }
])
display(risk_comparison.style.format({c: "{:.2%}" for c in risk_comparison.columns if c != "Confidence"}))

rolling_risk = compute_rolling_var_es(port_returns, window=60, alpha=0.05)
fig_rvar = plot_rolling_var_es(rolling_risk)
fig_rvar.show()

monthly_matrix = compute_monthly_returns(port_returns)
fig_mth = plot_monthly_heatmap(monthly_matrix)
fig_mth.show()


## 11. Markowitz Efficient Frontier & Asset Allocation


In [ ]:
sim_portfolios, min_vol_port, max_sharpe_port = generate_markowitz_efficient_frontier(
    returns, list(weights.index), num_portfolios=3500
)

current_point = {
    "volatility": port_metrics["annualized_volatility"],
    "return": port_metrics["annualized_return"]
}

fig_ef = plot_efficient_frontier(sim_portfolios, min_vol_port, max_sharpe_port, current_point)
fig_ef.show()


## 12. Stress Testing & Correlated Monte Carlo Simulation
Deterministic scenarios evaluate portfolio loss across parallel market shocks and concentrated idiosyncratic shocks.
The correlated Monte Carlo engine models joint adverse shocks incorporating empirical asset correlation.


In [ ]:
parallel_stress = run_deterministic_stress(weights, portfolio_value=100_000)
idiosyncratic_stress = run_idiosyncratic_stress(weights, target_ticker="NVDA", portfolio_value=100_000)

display(parallel_stress.style.format({"shock_pct": "{:.0%}", "dollar_loss": "${:,.0f}", "portfolio_loss_pct": "{:.2%}"}))
display(idiosyncratic_stress.style.format({"target_shock_pct": "{:.0%}", "dollar_loss": "${:,.0f}", "portfolio_loss_pct": "{:.2%}"}))

fig_stress = plot_stress_test_scenarios(parallel_stress, idiosyncratic_stress)
fig_stress.show()

mc_results = run_correlated_monte_carlo(returns, weights, n_sim=30000, annual_event_prob=0.02, shock_mean=-0.06, shock_sd=0.03)
print(f"Monte Carlo Mean Loss: {mc_results['mean_loss']:.3%}")
print(f"95% Scenario VaR: {mc_results['var_95']:.3%} | 95% Scenario ES: {mc_results['es_95']:.3%}")
print(f"99% Scenario VaR: {mc_results['var_99']:.3%} | 99% Scenario ES: {mc_results['es_99']:.3%}")

fig_mc = plot_monte_carlo_loss_distribution(mc_results["loss_distribution"], mc_results["var_95"], mc_results["es_95"])
fig_mc.show()


## 13. Research Limitations & Governance Roadmap
1. **Identification:** Single-firm event studies cannot separate CEO-specific reputational transmission from concurrent macroeconomic, semiconductor cycle, or earnings news.
2. **Event Verification:** Unverified news dates must not be entered into econometric models without primary documentation.
3. **Distributional Assumptions:** Monte Carlo scenario probabilities ($p=2\%$) and shock distributions are hypothetical stress parameters, not unconditional forecasts.
4. **Future Extensions:** High-frequency intraday tick-level event analysis, options-implied volatility surfaces, and multi-firm panel difference-in-differences regressions.
